In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from PIL import Image
from torchvision import models, transforms

In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


path set up 

In [63]:
import os

drive_root = "/content/drive/MyDrive"

matches = []
for root, dirs, files in os.walk(drive_root):
    if "scenario23_img_beam_train.csv" in files:
        matches.append(root)

print("Candidate folders:")
for m in matches:
    print(m)

Candidate folders:
/content/drive/MyDrive/Image beam


In [64]:
AUTHOR_ROOT = "/content/drive/MyDrive/Image beam"

In [65]:
train_csv_author = os.path.join(AUTHOR_ROOT, "scenario23_img_beam_train.csv")
val_csv_author   = os.path.join(AUTHOR_ROOT, "scenario23_img_beam_val.csv")
test_csv_author  = os.path.join(AUTHOR_ROOT, "scenario23_img_beam_test.csv")

print(os.path.exists(train_csv_author), train_csv_author)
print(os.path.exists(val_csv_author), val_csv_author)
print(os.path.exists(test_csv_author), test_csv_author)

True /content/drive/MyDrive/Image beam/scenario23_img_beam_train.csv
True /content/drive/MyDrive/Image beam/scenario23_img_beam_val.csv
True /content/drive/MyDrive/Image beam/scenario23_img_beam_test.csv


In [66]:
zip_path = "/content/drive/MyDrive/scenario23_dev_w_resources.zip"
print("Exists:", os.path.exists(zip_path))

Exists: True


In [67]:
import zipfile

extract_dir = "/content/dataset"

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_dir)

print("Unzipped to:", extract_dir)

Unzipped to: /content/dataset


In [68]:
for root, dirs, files in os.walk(extract_dir):
    if "scenario23" in root.lower():
        print(root)

/content/dataset/scenario23_dev
/content/dataset/scenario23_dev/unit2
/content/dataset/scenario23_dev/unit2/altitude
/content/dataset/scenario23_dev/unit2/distance
/content/dataset/scenario23_dev/unit2/height
/content/dataset/scenario23_dev/unit2/y_speed
/content/dataset/scenario23_dev/unit2/pitch
/content/dataset/scenario23_dev/unit2/speed
/content/dataset/scenario23_dev/unit2/roll
/content/dataset/scenario23_dev/unit2/z_speed
/content/dataset/scenario23_dev/unit2/x_speed
/content/dataset/scenario23_dev/unit2/GPS_data
/content/dataset/scenario23_dev/unit1
/content/dataset/scenario23_dev/unit1/mmWave_data
/content/dataset/scenario23_dev/unit1/camera_data
/content/dataset/scenario23_dev/unit1/GPS_data
/content/dataset/scenario23_dev/resources
/content/dataset/scenario23_dev/resources/bbox_labels_final


check zip in a better approach 

In [69]:
drive_root = "/content/drive/MyDrive"

zip_matches = []
for root, dirs, files in os.walk(drive_root):
    for f in files:
        if f.lower().endswith(".zip") and "scenario23" in f.lower():
            zip_matches.append(os.path.join(root, f))

print("Candidate ZIP files:")
for z in zip_matches:
    print(z)

Candidate ZIP files:
/content/drive/MyDrive/scenario23_dev_w_resources.zip


In [70]:
ZIP_PATH = "/content/drive/MyDrive/scenario23_dev_w_resources.zip"   # replace this
print("ZIP exists:", os.path.exists(ZIP_PATH), ZIP_PATH)

ZIP exists: True /content/drive/MyDrive/scenario23_dev_w_resources.zip


In [71]:
EXTRACT_ROOT = "/content/dataset"
os.makedirs(EXTRACT_ROOT, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_ROOT)

print("Unzip done")

Unzip done


detect real dataroot after unzip 

In [72]:
candidate_roots = []
for root, dirs, files in os.walk(EXTRACT_ROOT):
    if "scenario23.csv" in files:
        candidate_roots.append(root)

print("Candidate DATA_ROOT folders:")
for c in candidate_roots:
    print(c)

Candidate DATA_ROOT folders:
/content/dataset/scenario23_dev


In [73]:
DATA_ROOT = "/content/dataset/scenario23_dev"   # update if needed
print("Exists:", os.path.exists(DATA_ROOT))

Exists: True


build cell

In [84]:
# author image csv paths
train_csv_author = os.path.join(AUTHOR_ROOT, "scenario23_img_beam_train.csv")
val_csv_author   = os.path.join(AUTHOR_ROOT, "scenario23_img_beam_val.csv")
test_csv_author  = os.path.join(AUTHOR_ROOT, "scenario23_img_beam_test.csv")

# build image filename -> full path map from extracted dataset
image_map = {}
for root, dirs, files in os.walk(DATA_ROOT):
    for f in files:
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            image_map[f] = os.path.join(root, f)

print("Indexed images:", len(image_map))

def fix_img_csv_paths(csv_path, out_path):
    dfc = pd.read_csv(csv_path).copy()
    path_col = dfc.columns[1]

    dfc[path_col] = dfc[path_col].apply(
        lambda x: image_map.get(os.path.basename(str(x).strip()), str(x))
    )

    dfc.to_csv(out_path, index=False)
    print("Saved:", out_path)
    return dfc

fix_img_csv_paths(train_csv_author, train_csv_fixed)
fix_img_csv_paths(val_csv_author, val_csv_fixed)
fix_img_csv_paths(test_csv_author, test_csv_fixed)

Indexed images: 11387
Saved: /content/drive/MyDrive/scenario23_fixed_csv/scenario23_img_beam_train_fixed.csv
Saved: /content/drive/MyDrive/scenario23_fixed_csv/scenario23_img_beam_val_fixed.csv
Saved: /content/drive/MyDrive/scenario23_fixed_csv/scenario23_img_beam_test_fixed.csv


,index,unit1_rgb,unit1_beam
0,8849,/content/dataset/scenario23_dev/unit1/camera_d...,17
1,4597,/content/dataset/scenario23_dev/unit1/camera_d...,20
2,3236,/content/dataset/scenario23_dev/unit1/camera_d...,19
3,7657,/content/dataset/scenario23_dev/unit1/camera_d...,15
4,9776,/content/dataset/scenario23_dev/unit1/camera_d...,17
...,...,...,...
1134,7814,/content/dataset/scenario23_dev/unit1/camera_d...,25
1135,10956,/content/dataset/scenario23_dev/unit1/camera_d...,2
1136,906,/content/dataset/scenario23_dev/unit1/camera_d...,12
1137,5193,/content/dataset/scenario23_dev/unit1/camera_d...,9


In [85]:
train_csv_fixed = "/content/scenario23_img_beam_train_fixed.csv"
val_csv_fixed   = "/content/scenario23_img_beam_val_fixed.csv"
test_csv_fixed  = "/content/scenario23_img_beam_test_fixed.csv"

if os.path.exists(train_csv_fixed) and os.path.exists(val_csv_fixed) and os.path.exists(test_csv_fixed):
    print("Using existing fixed CSV files")
else:
    print("Building fixed CSV files from author CSV + extracted dataset")

    image_map = {}
    for root, dirs, files in os.walk(DATA_ROOT):
        for f in files:
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                image_map[f] = os.path.join(root, f)

    print("Indexed images:", len(image_map))

    def fix_img_csv_paths(csv_path, out_path):
        dfc = pd.read_csv(csv_path).copy()
        path_col = dfc.columns[1]
        dfc[path_col] = dfc[path_col].apply(
            lambda x: image_map.get(os.path.basename(str(x).strip()), str(x))
        )
        dfc.to_csv(out_path, index=False)
        print("Saved:", out_path)
        return dfc

    fix_img_csv_paths(train_csv_author, train_csv_fixed)
    fix_img_csv_paths(val_csv_author, val_csv_fixed)
    fix_img_csv_paths(test_csv_author, test_csv_fixed)

Using existing fixed CSV files


In [86]:
fixed_root = "/content/drive/MyDrive/scenario23_fixed_csv"
os.makedirs(fixed_root, exist_ok=True)

train_csv_fixed = os.path.join(fixed_root, "scenario23_img_beam_train_fixed.csv")
val_csv_fixed   = os.path.join(fixed_root, "scenario23_img_beam_val_fixed.csv")
test_csv_fixed  = os.path.join(fixed_root, "scenario23_img_beam_test_fixed.csv")

In [87]:
train_df = pd.read_csv(train_csv_fixed)
val_df   = pd.read_csv(val_csv_fixed)
test_df  = pd.read_csv(test_csv_fixed)

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)
print(train_df.head())

Train: (6832, 3)
Val  : (3416, 3)
Test : (1139, 3)
   index                                          unit1_rgb  unit1_beam
0   3532  /content/dataset/scenario23_dev/unit1/camera_d...          17
1   2224  /content/dataset/scenario23_dev/unit1/camera_d...          14
2   9416  /content/dataset/scenario23_dev/unit1/camera_d...          17
3   8510  /content/dataset/scenario23_dev/unit1/camera_d...          20
4   6877  /content/dataset/scenario23_dev/unit1/camera_d...          17


quick image path check 

In [88]:
for p in train_df.iloc[:10, 1].tolist():
    print(p, os.path.exists(str(p)))

/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_3532_17_08_22.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_2224_17_04_35.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_10033_17_56_07.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_9127_17_53_50.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_7494_17_48_19.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_3825_17_09_10.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_2258_17_04_41.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_2121_17_04_20.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_11899_18_02_04.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_8210_17_50_08.jpg True


dataset

In [89]:
class ImageBeamDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path).reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row.iloc[1]
        label = int(row.iloc[2])

        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)

        return img, label

In [90]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

In [91]:
batch_size = 32
val_batch_size = 32

train_loader = DataLoader(
    ImageBeamDataset(train_csv_fixed, transform=train_transform),
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    ImageBeamDataset(val_csv_fixed, transform=test_transform),
    batch_size=val_batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    ImageBeamDataset(test_csv_fixed, transform=test_transform),
    batch_size=val_batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [92]:
train_df = pd.read_csv(train_csv_fixed)

print(train_df.head())

for p in train_df.iloc[:10, 1].tolist():
    print(p, os.path.exists(str(p)))

   index                                          unit1_rgb  unit1_beam
0   3532  /content/dataset/scenario23_dev/unit1/camera_d...          17
1   2224  /content/dataset/scenario23_dev/unit1/camera_d...          14
2   9416  /content/dataset/scenario23_dev/unit1/camera_d...          17
3   8510  /content/dataset/scenario23_dev/unit1/camera_d...          20
4   6877  /content/dataset/scenario23_dev/unit1/camera_d...          17
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_3532_17_08_22.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_2224_17_04_35.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_10033_17_56_07.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_9127_17_53_50.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_7494_17_48_19.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_3825_17_09_10.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_2258_17_0

sanity check 

In [94]:
train_df = pd.read_csv(train_csv_fixed)
print(train_df.head())

for p in train_df.iloc[:10, 1].tolist():
    print(p, os.path.exists(str(p)))

   index                                          unit1_rgb  unit1_beam
0   3532  /content/dataset/scenario23_dev/unit1/camera_d...          17
1   2224  /content/dataset/scenario23_dev/unit1/camera_d...          14
2   9416  /content/dataset/scenario23_dev/unit1/camera_d...          17
3   8510  /content/dataset/scenario23_dev/unit1/camera_d...          20
4   6877  /content/dataset/scenario23_dev/unit1/camera_d...          17
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_3532_17_08_22.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_2224_17_04_35.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_10033_17_56_07.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_9127_17_53_50.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_7494_17_48_19.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_3825_17_09_10.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_2258_17_0

In [93]:
imgs, labels = next(iter(train_loader))
print(imgs.shape, labels.shape)
print(labels[:10])

torch.Size([32, 3, 224, 224]) torch.Size([32])
tensor([19, 17, 15, 16, 15, 16, 15, 17, 15, 15])


In [95]:
def evaluate_topk(model, loader, device, ks=(1, 2, 3, 5)):
    model.eval()
    total = 0
    correct = {k: 0 for k in ks}

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(imgs)
            max_k = max(ks)
            _, pred = torch.topk(outputs, k=max_k, dim=1)
            pred = pred.t()

            total += labels.size(0)
            for k in ks:
                correct[k] += pred[:k].eq(labels.view(1, -1)).sum().item()

    return {f"top{k}": 100.0 * correct[k] / total for k in ks}

train the epoch 

In [96]:
def train_model(
    model,
    train_loader,
    val_loader,
    device,
    epochs=10,
    lr=1e-4,
    weight_decay=1e-4,
    milestones=(5, 8),
    save_path="/content/drive/MyDrive/best_model.pth"
):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=list(milestones), gamma=0.1)

    best_top1 = -1
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        total_train = 0

        for step, (imgs, labels) in enumerate(train_loader, start=1):
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            bs = labels.size(0)
            running_loss += loss.item() * bs
            total_train += bs

            if step % 50 == 0:
                print(f"Epoch {epoch:02d} Step {step}/{len(train_loader)} Loss {loss.item():.4f}")

        scheduler.step()

        train_loss = running_loss / total_train
        val_metrics = evaluate_topk(model, val_loader, device, ks=(1, 2, 3, 5))

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            **val_metrics
        })

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Top1: {val_metrics['top1']:.2f} | "
            f"Top2: {val_metrics['top2']:.2f} | "
            f"Top3: {val_metrics['top3']:.2f} | "
            f"Top5: {val_metrics['top5']:.2f}"
        )

        if val_metrics["top1"] > best_top1:
            best_top1 = val_metrics["top1"]
            torch.save(copy.deepcopy(model.state_dict()), save_path)
            print("Saved best model")

    return pd.DataFrame(history)

In [97]:
vgg_model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
vgg_model.classifier[6] = nn.Linear(vgg_model.classifier[6].in_features, 64)
vgg_model = vgg_model.to(device)

hist_vgg = train_model(
    model=vgg_model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=10,
    lr=1e-4,
    weight_decay=1e-4,
    milestones=(5, 8),
    save_path="/content/drive/MyDrive/best_vgg16.pth"
)

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:07<00:00, 74.4MB/s] 


Epoch 01 Step 50/214 Loss 1.3011
Epoch 01 Step 100/214 Loss 0.4364
Epoch 01 Step 150/214 Loss 0.7944
Epoch 01 Step 200/214 Loss 0.9262
Epoch 01 | Train Loss: 1.1637 | Top1: 83.84 | Top2: 96.66 | Top3: 98.80 | Top5: 99.41
Saved best model
Epoch 02 Step 50/214 Loss 0.6557
Epoch 02 Step 100/214 Loss 0.6541
Epoch 02 Step 150/214 Loss 0.6087
Epoch 02 Step 200/214 Loss 0.4247
Epoch 02 | Train Loss: 0.4904 | Top1: 84.60 | Top2: 97.75 | Top3: 99.12 | Top5: 99.62
Saved best model
Epoch 03 Step 50/214 Loss 0.2600
Epoch 03 Step 100/214 Loss 0.5654
Epoch 03 Step 150/214 Loss 0.4401
Epoch 03 Step 200/214 Loss 0.2814
Epoch 03 | Train Loss: 0.4210 | Top1: 85.92 | Top2: 97.19 | Top3: 99.24 | Top5: 99.59
Saved best model
Epoch 04 Step 50/214 Loss 0.4902
Epoch 04 Step 100/214 Loss 0.3724
Epoch 04 Step 150/214 Loss 0.5293
Epoch 04 Step 200/214 Loss 0.6612
Epoch 04 | Train Loss: 0.3826 | Top1: 87.35 | Top2: 98.19 | Top3: 99.36 | Top5: 99.59
Saved best model
Epoch 05 Step 50/214 Loss 0.1299
Epoch 05 Step 1

In [98]:
vgg_model.load_state_dict(torch.load("/content/drive/MyDrive/best_vgg16.pth", map_location=device))
vgg_model = vgg_model.to(device)

test_metrics_vgg = evaluate_topk(vgg_model, test_loader, device, ks=(1, 2, 3, 5))
print("VGG16 Test metrics:", test_metrics_vgg)

VGG16 Test metrics: {'top1': 88.49868305531167, 'top2': 97.98068481123792, 'top3': 99.82440737489026, 'top5': 99.82440737489026}


In [34]:
def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params

total_params, trainable_params = count_parameters(vgg_model)

param_size_mb = sum(p.numel() * p.element_size() for p in vgg_model.parameters()) / (1024 ** 2)
buffer_size_mb = sum(b.numel() * b.element_size() for b in vgg_model.buffers()) / (1024 ** 2)
model_size_mb = param_size_mb + buffer_size_mb

print(f"Total params     : {total_params:,}")
print(f"Trainable params : {trainable_params:,}")
print(f"Model size (MB)  : {model_size_mb:.2f}")

Total params     : 134,522,752
Trainable params : 134,522,752
Model size (MB)  : 513.16


In [35]:
def measure_latency(model, loader, device, warmup_batches=5, measure_batches=20):
    model.eval()
    times = []

    # warmup
    with torch.no_grad():
        for i, (imgs, labels) in enumerate(loader):
            if i >= warmup_batches:
                break
            imgs = imgs.to(device, non_blocking=True)
            _ = model(imgs)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    # measure
    with torch.no_grad():
        for i, (imgs, labels) in enumerate(loader):
            if i >= measure_batches:
                break

            imgs = imgs.to(device, non_blocking=True)

            if torch.cuda.is_available():
                torch.cuda.synchronize()
            start = time.time()

            _ = model(imgs)

            if torch.cuda.is_available():
                torch.cuda.synchronize()
            end = time.time()

            times.append(end - start)

    avg_batch_time = np.mean(times)
    std_batch_time = np.std(times)
    batch_size_now = next(iter(loader))[0].shape[0]
    avg_img_time = avg_batch_time / batch_size_now
    throughput = batch_size_now / avg_batch_time

    return {
        "avg_batch_time_sec": avg_batch_time,
        "std_batch_time_sec": std_batch_time,
        "avg_img_time_ms": avg_img_time * 1000,
        "throughput_img_per_sec": throughput
    }

latency_vgg = measure_latency(vgg_model, test_loader, device)
latency_vgg

{'avg_batch_time_sec': np.float64(0.13728322982788085),
 'std_batch_time_sec': np.float64(0.0017125289058490863),
 'avg_img_time_ms': np.float64(4.290100932121277),
 'throughput_img_per_sec': np.float64(233.09474900991233)}

In [36]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    vgg_model.eval()
    imgs, labels = next(iter(test_loader))
    imgs = imgs.to(device, non_blocking=True)

    with torch.no_grad():
        _ = vgg_model(imgs)

    torch.cuda.synchronize()
    peak_mem_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
    print(f"Peak GPU memory during one inference batch: {peak_mem_mb:.2f} MB")
else:
    print("CUDA not available")

Peak GPU memory during one inference batch: 3612.52 MB


In [37]:
vgg_efficiency = pd.DataFrame([{
    "Model": "VGG16",
    "Total Params": total_params,
    "Trainable Params": trainable_params,
    "Model Size (MB)": model_size_mb,
    "Avg Batch Time (s)": latency_vgg["avg_batch_time_sec"],
    "Avg Image Time (ms)": latency_vgg["avg_img_time_ms"],
    "Throughput (img/s)": latency_vgg["throughput_img_per_sec"],
    "Top-1": test_metrics_vgg["top1"],
    "Top-2": test_metrics_vgg["top2"],
    "Top-3": test_metrics_vgg["top3"],
    "Top-5": test_metrics_vgg["top5"],
}])

vgg_efficiency

,Model,Total Params,Trainable Params,Model Size (MB),Avg Batch Time (s),Avg Image Time (ms),Throughput (img/s),Top-1,Top-2,Top-3,Top-5
0,VGG16,134522752,134522752,513.163574,0.137283,4.290101,233.094749,88.586479,98.507463,99.824407,99.824407


vit model 

In [ ]:
vit_model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
vit_model.heads.head = nn.Linear(vit_model.heads.head.in_features, 64)
vit_model = vit_model.to(device)

hist_vit = train_model(
    model=vit_model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=10,
    lr=1e-4,
    weight_decay=1e-4,
    milestones=(5, 8),
    save_path="/content/drive/MyDrive/best_vit_b16.pth"
)

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:01<00:00, 178MB/s] 


In [ ]:
vit_model.load_state_dict(torch.load("/content/drive/MyDrive/best_vit_b16.pth", map_location=device))
vit_model = vit_model.to(device)

test_metrics_vit = evaluate_topk(vit_model, test_loader, device, ks=(1, 2, 3, 5))
print("ViT-B/16 Test metrics:", test_metrics_vit)